# A Tiny Convolutional Network

## The Convolution

$$
(x \star \kappa)_{rc} = b + \sum_{u=-2}^{2} \sum_{v=-2}^{2} \kappa_{uv} \, x_{r+u,\, c+v}
$$

## The Network

### Flatten + Linear: the Output Layer

$$
o = z \, w_o + b_o = \sum_{j=1}^{64k} z_j w_{o,j} + b_o
$$

In [ ]:
import io
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")
ARCHIVE = DATA_DIR / "UTKFace.tar.gz"
CACHE = DATA_DIR / "utkface_32.npz"
ARCHIVE_URL = "https://huggingface.co/datasets/py97/UTKFace-Cropped/resolve/main/UTKFace.tar.gz"
IMAGE_SIZE = 32

## Pooling

$$
\text{max-pool}_{rc} = \max_{0 \le u, v < s} f_{sr + u,\; sc + v}, \qquad
\text{avg-pool}_{rc} = \frac{1}{s^2} \sum_{u, v = 0}^{s-1} f_{sr + u,\; sc + v}
$$

### A Toy Example

## Data

In [ ]:
def build_cache():
    # Read the faces straight from the archive, convert to grayscale and shrink
    DATA_DIR.mkdir(exist_ok=True)
    if not ARCHIVE.exists():
        urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE)

    images, ages, genders = [], [], []
    with tarfile.open(ARCHIVE) as tar:
        for member in tar:
            if not (member.isfile() and member.name.endswith(".jpg")):
                continue
            fields = member.name.split("/")[-1].split("_")
            try:
                age, gender = int(fields[0]), int(fields[1])
            except ValueError:
                continue
            if gender not in (0, 1):
                continue
            image = Image.open(io.BytesIO(tar.extractfile(member).read()))
            image = image.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
            images.append(np.asarray(image, dtype=np.uint8))
            ages.append(age)
            genders.append(gender)
    np.savez_compressed(CACHE, images=np.stack(images), age=ages, gender=genders)

if not CACHE.exists():
    build_cache()

data = np.load(CACHE)
keep = data["age"] <= 80
images = data["images"][keep]          # (n, 32, 32), uint8
age = data["age"][keep]
female = data["gender"][keep].astype(np.float32)

idx_train, idx_rest = train_test_split(np.arange(len(age)), test_size=0.3, random_state=0, stratify=female)
idx_val, idx_test = train_test_split(idx_rest, test_size=0.5, random_state=0, stratify=female[idx_rest])

pixels = images.astype(np.float32) / 255
pixel_mean, pixel_std = pixels[idx_train].mean(), pixels[idx_train].std()

def standardize_global(batch):
    # one mean and one standard deviation for all pixels; returns (n, 1, 32, 32)
    return torch.tensor((batch - pixel_mean) / pixel_std).unsqueeze(1)

X_cnn = standardize_global(pixels)
y_female = torch.tensor(female)
print("train / val / test:", len(idx_train), len(idx_val), len(idx_test))

## Training

In [ ]:
def fit(model, X, target, n_epochs=15, learning_rate=3e-3, weight_decay=0.0, batch_size=128, seed=0):
    # Mini-batch Adam; returns the model with the best validation loss and the learning curves
    loss_fn = nn.BCEWithLogitsLoss()
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    X_train, y_train = X[idx_train], target[idx_train]
    X_val, y_val = X[idx_val], target[idx_val]

    history = {"train": [], "val": []}
    best_loss, best_state = float("inf"), None
    for epoch in range(n_epochs):
        model.train()
        order = torch.randperm(len(idx_train))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            optimizer.zero_grad()
            loss_fn(model(X_train[batch]).squeeze(1), y_train[batch]).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            history["train"].append(loss_fn(model(X_train).squeeze(1), y_train).item())
            history["val"].append(loss_fn(model(X_val).squeeze(1), y_val).item())
        if history["val"][-1] < best_loss:
            best_loss = history["val"][-1]
            best_state = {name: value.clone() for name, value in model.state_dict().items()}
            history["best_epoch"] = epoch

    model.load_state_dict(best_state)
    return model, history

def accuracy(model, X):
    model.eval()
    with torch.no_grad():
        return ((model(X).squeeze(1) > 0).float() == y_female[idx_test]).float().mean().item()

In [ ]:
class TinyCNN(nn.Module):
    # one convolutional layer with n_kernels kernels, max-pooling and a linear output (the logit)
    def __init__(self, n_kernels=2, kernel_size=5, pool_size=4, image_size=IMAGE_SIZE):
        super().__init__()
        self.conv = nn.Conv2d(1, n_kernels, kernel_size, padding=kernel_size // 2)
        self.pool = nn.MaxPool2d(pool_size)
        self.output = nn.Linear(n_kernels * (image_size // pool_size) ** 2, 1)

    def feature_maps(self, x):
        # (n, 1, 32, 32) -> (n, n_kernels, 32, 32)
        return torch.relu(self.conv(x))

    def forward(self, x):
        pooled = self.pool(self.feature_maps(x))     # (n, n_kernels, 8, 8)
        return self.output(pooled.flatten(start_dim=1))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
N_KERNELS = 2

torch.manual_seed(0)
cnn = TinyCNN(n_kernels=N_KERNELS)
initial_kernels = cnn.conv.weight.detach().clone()
print("parameters:", count_parameters(cnn))

cnn, hist_cnn = fit(cnn, X_cnn, y_female)
print(f"CNN test accuracy: {accuracy(cnn, X_cnn[idx_test]):.3f}")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(hist_cnn["train"], label="train")
ax.plot(hist_cnn["val"], label="validation")
ax.set(xlabel="epoch", ylabel="binary cross-entropy")
ax.legend()
plt.show()

In [ ]:
X_flat = torch.tensor((pixels.reshape(len(pixels), -1) - pixels[idx_train].reshape(len(idx_train), -1).mean(0))
                      / (pixels[idx_train].reshape(len(idx_train), -1).std(0) + 1e-6))

torch.manual_seed(0)
ann = nn.Sequential(nn.Linear(1024, 64), nn.ReLU(), nn.Linear(64, 1))
ann, _ = fit(ann, X_flat, y_female, learning_rate=1e-3, weight_decay=1e-3, n_epochs=60)

print(f"ANN test accuracy (65,729 parameters): {accuracy(ann, X_flat[idx_test]):.3f}")
print(f"CNN test accuracy ({count_parameters(cnn)} parameters)    : {accuracy(cnn, X_cnn[idx_test]):.3f}")

## What Did the Kernels Learn?

### Feature Maps

### Where Does the Output Layer Look?

## Robustness to Shifts

## Summary

## Exercises